In [28]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("Data/")

In [29]:
files = {
    "orders":         "olist_orders_dataset.csv",
    "items":          "olist_order_items_dataset.csv",
    "customers":      "olist_customers_dataset.csv",
    "products":       "olist_products_dataset.csv",
    "payments":       "olist_order_payments_dataset.csv",
    "reviews":        "olist_order_reviews_dataset.csv",
    "sellers":        "olist_sellers_dataset.csv",
    "geo":            "olist_geolocation_dataset.csv",
    "category_names": "product_category_name_translation.csv",
}

tables = {name: pd.read_csv(DATA_DIR / fname) for name, fname in files.items()}

for name, df in tables.items():
    print(f"{name:15} {df.shape[0]:10,} rows {df.shape[1]:>3} cols")

orders, items, customers = tables["orders"], tables["items"], tables["customers"]
products, payments, reviews = tables["products"], tables["payments"], tables["reviews"]

orders              99,441 rows   8 cols
items              112,650 rows   7 cols
customers           99,441 rows   5 cols
products            32,951 rows   9 cols
payments           103,886 rows   5 cols
reviews             99,224 rows   7 cols
sellers              3,095 rows   4 cols
geo              1,000,163 rows   5 cols
category_names          71 rows   2 cols


In [30]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [31]:
date_cols = {
    "orders":  ["order_purchase_timestamp", "order_approved_at",
                "order_delivered_carrier_date", "order_delivered_customer_date",
                "order_estimated_delivery_date"],
    "items":   ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],    
}

for table_name, cols in date_cols.items():
    df = tables[table_name]
    for col in cols:
        before = df[col].isna().sum()
        df[col] = pd.to_datetime(df[col], errors="coerce")
        after = df[col].isna().sum()
        print(f"{table_name}.{col}: missing before {before}, after {after}")




orders.order_purchase_timestamp: missing before 0, after 0
orders.order_approved_at: missing before 160, after 160
orders.order_delivered_carrier_date: missing before 1783, after 1783
orders.order_delivered_customer_date: missing before 2965, after 2965
orders.order_estimated_delivery_date: missing before 0, after 0
items.shipping_limit_date: missing before 0, after 0
reviews.review_creation_date: missing before 0, after 0
reviews.review_answer_timestamp: missing before 0, after 0


In [27]:
delivery_days = (orders["order_delivered_customer_date"]- orders["order_purchase_timestamp"]).dt.days
delivery_days.describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
dtype: float64